# Wstępna Eksploracyjna Analiza Danych ze Steam
Celem tej sekcji jest ustalenie wstępnych wymagań wobec zbioru danych przed jego pełnym pobraniem ze Steam API.

W projekcie celem jest stworzenie algorytmu wykrywającego sentyment oceny gry, uwzględniając sarkzam.

Wytrenowanie skutecznego modelu będzie wymagało przekroju różnych opinii, z różnych gatunków gier, o różnych wymogach technicznych. Powody opinii negatywnej czy sarkastycznej mogą wynikać zarówno z problemów z fabułą i mechaniką gry, ale również z braku optymalizacji gry dla słabszego sprzętu. Algorytm wytrenowany wyłącznie na recenzjach najpopularniejszych gier mogłoby skutkować z tendencyjnością algorytmu i wysoką liczebnością klasy False Negative.

Dodatkowo zdaniem autora najpopularniejsze gry mogą częściej mieć problemy z optymalizacją niż z samą rozgrywką, niż przeciętna gra. Gracze chętniej spróbują zagrać w ciekawą grę, która jest niezoptymalizowana na ich sprzęt, z nadzieją, że twórca gry po pewnym czasie wypuści aktualizację, która ustabilizuje ilość klatek na sekundę na ich sprzęcie. Natomiast, jeśli gra jest dobrze zoptymalizowana ale nudna gra nigdy nie trafi do listy najpopularniejszych gier.

Osobnym wyzwaniem jest ilość opinii w języku polskim. Sarkazm jest zjawiskiem bardzo specficznym dla każdego języka, dlatego do trenowania modelu autor wybierze wyłącznie opinie w języku polskim. Tym sposobem autor będzie mógł na koniec podejrzeć wyniki algorytmu i samodzielnie ocenić jego poprawne funkcjonowanie. Ograniczenie językowe oznacza, że skupienie się wyłącznie na mało popularnych grach może skutkować trudnością z uzbieraniem odpowiedniej ilości opinii i znacznie większą ilością zapytań do Steam API.

Zważając na powyższe ograniczenia autor zamierza najpierw podzielić dużą populację najczęściej granych gier podzielić na grupy według ich kategorii oceny Steam, a następnie z każdej grupy losowo wybrać podobną liczbę gier, aby cały zbiór recenzji posiadał około **??** tysięcy opinii.

## Przegląd najpopularniejszych gier na platformie Steam

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

In [5]:
url = "https://store.steampowered.com/search/?supportedlang=polish%2Cenglish&category1=998&ndl=1"
    
# Udajemy prawdziwą przeglądarkę (Header), żeby Steam nas nie zablokował
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept-Language': 'pl,en;q=0.9'
}

# 4. Wywołujemy zapytanie przekazując ZARÓWNO nagłówki, jak i ciasteczka
response = requests.get(url, headers=headers)

# Jeśli strona nie zwraca JSON-a, bierzemy surowy tekst HTML
html_content = response.text

print(html_content)

<!DOCTYPE html>
<html class=" responsive DesktopUI" lang="pl"  >
<head>
	<meta http-equiv="Content-Type" content="text/html; charset=UTF-8">
			<meta name="viewport" content="width=device-width,initial-scale=1">
		<meta name="theme-color" content="#171a21">
		<title>Wyszukiwanie Steam</title>
	<link rel="shortcut icon" href="/favicon.ico" type="image/x-icon">

	
	
	<link href="https://store.fastly.steamstatic.com/public/shared/css/motiva_sans.css?v=YzJgj1FjzW34&amp;l=polish&amp;_cdn=fastly" rel="stylesheet" type="text/css">
<link href="https://store.fastly.steamstatic.com/public/shared/css/shared_global.css?v=F0AmfT9E8goz&amp;l=polish&amp;_cdn=fastly" rel="stylesheet" type="text/css">
<link href="https://store.fastly.steamstatic.com/public/shared/css/buttons.css?v=BZhNEtESfYSJ&amp;l=polish&amp;_cdn=fastly" rel="stylesheet" type="text/css">
<link href="https://store.fastly.steamstatic.com/public/css/v6/store.css?v=QQDgnRsK7TU9&amp;l=polish&amp;_cdn=fastly" rel="stylesheet" type="text/cs

In [6]:
soup = BeautifulSoup(html_content, 'html.parser')
# print(soup)
search_results = soup.find_all('a', class_='search_result_row')
print(search_results)

[<a class="search_result_row ds_collapse_flag" data-ds-appid="730" data-ds-crtrids="[4]" data-ds-descids="[2,5]" data-ds-itemkey="App_730" data-ds-steam-deck-compat-handled="true" data-ds-tagids="[1663,1774,3859,3878,19,5711,5055]" data-gpnav="item" data-search-page="1" href="https://store.steampowered.com/app/730/CounterStrike_2/?snr=1_7_7_230_150_1" onmouseout="HideGameHover( this, event, 'global_hover' )" onmouseover="GameHover( this, event, 'global_hover', {&quot;type&quot;:&quot;app&quot;,&quot;id&quot;:730,&quot;public&quot;:1,&quot;v6&quot;:1} );">
<div class="search_capsule"><img src="https://shared.fastly.steamstatic.com/store_item_assets/steam/apps/730/capsule_231x87.jpg?t=1749053861"/></div>
<div class="responsive_search_name_combined">
<div class="search_name ellipsis">
<span class="title">Counter-Strike 2</span>
</div>
<div class="search_platforms">
<span class="platform_img win"></span><span class="platform_img linux"></span> </div>
<div class="search_released responsive_

In [8]:
for item in search_results:
    # try:
    # Wyciągamy ID aplikacji (Steam trzyma je w atrybucie 'data-ds-appid')
    app_id = item.get('data-ds-appid')
    
    # Wyciągamy nazwę gry
    name = item.find('span', class_='title').text.strip()
    
    # Szukamy sekcji z opiniami (ma klasą 'search_review_summary')
    review_div = item.find('span', class_='search_review_summary')
    
    if review_div:
        # Wyciągamy ukryty opis tekstowy, np. "Mixed - 45% of the 1,200 user reviews..."
        review_html = review_div.get('data-tooltip-html', '')
        
        # Używamy Wyrażeń Regularnych (Regex), żeby wyciągnąć procenty i liczbę opinii
        # Szukamy wzorca: "XX% of the XX,XX user reviews"
        percent_grp = re.findall(r'(\d\d)%', review_html)
        reviews_grp = re.findall(r'z ([\d,]+) recenzji', review_html)
        
        review_percent = int(percent_grp[0]) if percent_grp else None
        
        # Usuwamy przecinki z liczby opinii, np. 1,200 -> 1200
        review_count = int(reviews_grp[0].replace(',', '')) if reviews_grp else 0
        print(f"{app_id} - {name} - {review_percent}% - {review_count}")
    # except Exception as e:
    #     # Jeśli jedna gra rzuci błędem podczas parsowania, idziemy do kolejnej
    #     continue

730 - Counter-Strike 2 - 87% - 501118
2483190 - Forza Horizon 6 - 90% - 857
1962700 - Subnautica 2 - 93% - 1848
2215200 - LEGO® Batman™: Dziedzictwo Mrocznego Rycerza - 96% - 3417
227300 - Euro Truck Simulator 2 - 97% - 54780
264710 - Subnautica - 98% - 6135
3892270 - Gamble With Your Friends - 92% - 339
1174180 - Red Dead Redemption 2 - 95% - 15969
3041230 - Windrose - 92% - 729
3321460 - Crimson Desert - 86% - 796
3105440 - Heroes of Might and Magic: Olden Era - 92% - 651
381210 - Dead by Daylight - 86% - 17180
3405690 - EA SPORTS FC™ 26 - 51% - 3232
2183900 - Warhammer 40,000: Space Marine 2 - 91% - 2125
848450 - Subnautica: Below Zero - 93% - 2111
1142710 - Total War: WARHAMMER III - 79% - 1276
2344520 - Diablo® IV - 80% - 1339
230410 - Warframe - 93% - 9747
2300320 - Farming Simulator 25 - 90% - 3235
236390 - War Thunder - 82% - 24844
3472040 - NBA 2K26 - 81% - 238
2767030 - Marvel Rivals - 78% - 4503
2605790 - Deep Rock Galactic: Rogue Core - 59% - 8255
3240220 - Grand Theft Auto